# Optuna

### Used for faster and best Hyperparameter tuning. 

---

## Key Term :

### 1. Study : It is an Optimization Session that encompasses multiple Trails.
- It essentially a collection of trails aimed at optimizing the objective function.
- Ex. A study to find the best hyperparameter for XGBoost Model.

### 2. Trail : A Trail is a single iteration of the optimization process wherea specific set of hyperparameter is evaluated.
- Each trail runs specific objective function with distinct set of hyperparameters.
- Ex. One trail could involve training a RandomForest Model with learning Rate = 0.01 and max_depth = 5.

### 3. Trail Parameter : These are the specific Hyperparameter values selected during the trail.
- Each trail have unique combination of hyperparameter that are evaluated to see how they impact the objective funciton.
- Ex. for one trail hyperparameters are {'learning_rate' = 0.01, 'max_depth' = 5} for second trail {'learning_rate' = 0.001, 'max_depth' = 7}

### 4. Objective Function : The objective function is the function to be optimized during the hyperparameter tuning.
- It takes hyperparameter as inputs and return a value that Optuna trys to optimize.
- Ex. In a Classification task, the objective function could be the cross-entropy loss, which Optuna seeks to minimize.

### 5. Sampler: A Samper is a algorithm that suggest which hyperparameter should be evaluated next.
- Optuna uses the Tree-structured Parzen Estimator(TSE) by defualt, but it also supports other sampling methods like Random Search or even custom sampler.
- Ex. TPE suggests promising areas of hyperparameter space, focusing on regions that are likely to yield better results.

In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI','DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# For Single Model Hyperparameter tuning

In [4]:
import optuna

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

c:\Users\Keval\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-25 21:02:18,103] A new study created in memory with name: no-name-456da6ec-025d-4bba-add7-ecdcf0af7afc
[I 2026-08-25 21:02:18,661] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 91, 'max_depth': 11}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-08-25 21:02:19,342] Trial 1 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 137, 'max_depth': 5}. Best is trial 1 with value: 0.7690875232774674.
[I 2026-08-25 21:02:20,567] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 193, 'max_depth': 20}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-25 21:02:21,710] Trial 3 finished with value: 0.77094972067

In [6]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7821229050279329
Best hyperparameters: {'n_estimators': 104, 'max_depth': 12}


In [7]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)

print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.76


### Optuna also have different type of Samplers Like RandomSampler, GridSampler, TPESampler, etc.

# Random Sampler

In [8]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-08-25 20:53:34,769] A new study created in memory with name: no-name-17d4a6ca-ba39-4334-85a7-ff0e20d02236
[I 2026-08-25 20:53:35,457] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 58, 'max_depth': 20}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-25 20:53:35,998] Trial 1 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 98, 'max_depth': 9}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-25 20:53:36,400] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 71, 'max_depth': 16}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-25 20:53:36,929] Trial 3 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 103, 'max_depth': 4}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-25 20:53:37,933] Trial 4 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 183, 'max_depth': 14}. Best is trial 2 with value: 0.774674115

In [9]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350094
Best hyperparameters: {'n_estimators': 70, 'max_depth': 17}


# Grid Sampler

In [10]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-08-25 20:55:02,078] A new study created in memory with name: no-name-4b257c95-50f2-4e92-b5a0-fe762834de67
[I 2026-08-25 20:55:02,587] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-25 20:55:03,448] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-25 20:55:03,758] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-25 20:55:04,318] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-25 20:55:04,869] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [11]:
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


# Optuna Visualization

In [5]:
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [6]:
plot_optimization_history(study).show()

In [7]:
plot_parallel_coordinate(study).show()

In [8]:
plot_slice(study).show()

In [9]:
plot_contour(study).show()

In [10]:
plot_param_importances(study).show()

# Optimizing Multiple Models

In [11]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [12]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-08-25 21:04:53,750] A new study created in memory with name: no-name-a7755801-64ad-4e7d-bc7f-e849d69d5070
[I 2026-08-25 21:04:55,692] Trial 0 finished with value: 0.7374301675977654 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 196, 'learning_rate': 0.09672681895497999, 'max_depth': 19, 'min_samples_split': 8, 'min_samples_leaf': 8}. Best is trial 0 with value: 0.7374301675977654.
[I 2026-08-25 21:04:55,750] Trial 1 finished with value: 0.7113594040968342 and parameters: {'classifier': 'SVM', 'C': 13.449010228054057, 'kernel': 'poly', 'gamma': 'auto'}. Best is trial 0 with value: 0.7374301675977654.
[I 2026-08-25 21:04:57,987] Trial 2 finished with value: 0.7225325884543761 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 212, 'learning_rate': 0.07429775621902175, 'max_depth': 18, 'min_samples_split': 6, 'min_samples_leaf': 6}. Best is trial 0 with value: 0.7374301675977654.
[I 2026-08-25 21:04:58,012] Trial 3 finished with value: 0.785847

In [13]:
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.15404998774800013, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [15]:
trail_df = study.trials_dataframe()

In [16]:
trail_df['params_classifier'].value_counts()

params_classifier
SVM                 79
GradientBoosting    11
RandomForest        10
Name: count, dtype: int64

In [17]:
trail_df.groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.744710
RandomForest        0.768156
SVM                 0.776065
Name: value, dtype: float64